# 03. Prompt-Based Image Editing (images.edit)

This notebook demonstrates **image editing guided only by a text prompt** — no mask — using
Azure OpenAI's `images.edit()` endpoint. It takes an existing product photo (`product_photo.png`)
and asks the model to restyle it into a "professional marketing visual" while keeping the product
itself recognizable, saving the result to `edited_product_photo.png`. This is the middle step in
the *generate → prompt-edit → mask-edit* progression in this section (02 → 03 → 04).

**Difficulty:** Beginner / Intermediate — builds directly on notebook 02, swapping
`images.generate()` for `images.edit()`.

## Prerequisites

**Python packages** (already in the repo's root `requirements.txt` — no extra install needed):
- `openai`, `python-dotenv`
- `base64`, `pathlib` — standard library

**Azure resources**
- The same Azure OpenAI **image model deployment** used in notebook 02 (e.g. `gpt-image-2`),
  since `images.edit()` is served by the same image-model family as `images.generate()`.

**Environment variables** — reuse the same names as notebook 02, added to the root `.env`:
```
AZURE_OPENAI_IMAGE_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_IMAGE_API_KEY=<your-api-key>
AZURE_OPENAI_IMAGE_DEPLOYMENT=gpt-image-2
```

**Input file:** `product_photo.png` must exist in the same folder as this notebook (it already
does, as a sample asset).

## What You'll Learn

- How `images.edit()` differs from `images.generate()` — it takes an *existing* image plus a
  prompt describing the desired change, instead of generating from nothing
- That prompt-only editing (no mask) lets the model potentially modify the **entire** image,
  relying purely on the prompt's wording (e.g. "keep the main product unchanged") to constrain
  what changes — a soft instruction, not a hard guarantee
- Why this matters as a contrast to notebook 04's **mask-based** editing, which gives you a hard,
  pixel-precise boundary on what can change

### Imports and setup

Same imports as notebook 02, plus `Path` is used up front here to define both the input and
output image paths as named variables.

💡 **Exam tip:** `images.edit()` and `images.generate()` are both part of the same Images API
surface in Azure OpenAI — know that editing requires an existing source image submitted as
binary file content (multipart), not a URL or base64 string, in the request.

🔄 **Alternatives:** N/A for imports — but functionally, this whole notebook is itself an
"alternative" to notebook 04: prompt-only edit vs. mask-based edit.

In [ ]:
import os
from pathlib import Path
import base64

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

### Configuration and file paths

- `endpoint`, `api_key`, `IMAGE_DEPLOYMENT_NAME` are read from the environment, same as
  notebook 02.
- `input_image_path` points at the existing photo to edit; `output_image_path` is where the
  edited result will be written.
- `prompt` is a multi-line instruction string describing the desired transformation. Note the
  instruction "Keep the main product unchanged" is a **request to the model**, not an enforced
  constraint — without a mask, the model is free to alter any pixel it judges relevant to
  fulfilling the prompt.

💡 **Exam tip:** For AI-102, understand that prompt-only image edits are inherently less
predictable/precise than mask-based edits — this is a key reason Azure OpenAI's Images API
supports an optional `mask` parameter at all (see notebook 04).

In [ ]:
endpoint = os.getenv("AZURE_OPENAI_IMAGE_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_IMAGE_API_KEY")
IMAGE_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_IMAGE_DEPLOYMENT", "gpt-image-2")

input_image_path = Path("product_photo.png")
output_image_path = Path("edited_product_photo.png")

prompt = """
Update this image so it looks like a professional marketing visual.
Keep the main product unchanged.
Improve the lighting, make the background cleaner, and give it a premium corporate style.
"""

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

### Calling images.edit()

- Opens the source image in binary mode and passes the open file object directly as `image=`.
- `client.images.edit(...)` sends the image + prompt to the model. Note there is **no `mask`
  argument** in this call — that's what makes this "prompt-based" editing.
- `size`, `n`, and `quality` behave the same as in `images.generate()`.

🔄 **Alternatives:** Add a `mask` argument (see notebook 04) to constrain edits to a specific
region instead of leaving the entire image open to change.

In [ ]:
with input_image_path.open("rb") as image_file:
    response = client.images.edit(
        model=IMAGE_DEPLOYMENT_NAME,
        image=image_file,
        prompt=prompt,
        size="1024x1024",
        n=1,
        quality="medium"
    )

### Decoding and saving the edited image

Same pattern as notebook 02: decode the base64 `b64_json` payload and write the raw bytes to
disk. Running this notebook overwrites `edited_product_photo.png` in the working directory
(the sample output already present in this folder).

In [ ]:
image_base64 = response.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

output_image_path.write_bytes(image_bytes)

print(f"Edited image saved to: {output_image_path}")

## Summary

This notebook sent an existing product photo plus a restyling prompt to `images.edit()` with no
mask, letting the model regenerate the image freely while trying to honor the "keep the product
unchanged" instruction. The result was saved to `edited_product_photo.png`. Because there's no
hard boundary on what can change, results can vary in how faithfully the original product is
preserved — which sets up the motivation for mask-based editing in notebook 04.

## Try It Yourself

1. **Easy:** Change the prompt to request a different style (e.g. "minimalist Scandinavian
   style" instead of "premium corporate style") and compare results.
2. **Intermediate:** Run the same prompt twice and compare how much the product itself changes
   between runs — this demonstrates the non-determinism/imprecision of prompt-only editing.
3. **Advanced:** Rewrite the prompt to be much more explicit about exactly which pixels/regions
   should NOT change (e.g. "do not alter anything within the central third of the image"), and
   evaluate whether wording alone can approximate what a mask guarantees structurally.